# Sen1Floods11: does terrain close the gap? S1-only vs S1 + JRC / DEM / slope / HAND

One notebook, Kaggle GPU T4, Internet ON. It answers one question with a controlled design:
**do terrain inputs (the ones DeepSARFlood uses) explain the gap between S1-only models (about 0.67-0.69) and the published 0.72?**

Pipeline
1. Download the 446 hand-labelled chips + official splits (public bucket). Fetch JRC seasonality, NASADEM elevation, slope and HAND for exactly those chip grids from **Earth Engine** (needs your EE project, see the next cell).
2. **Leave-one-event-out (LOEO)** screening on all 11 events (resnet34, fixed steps, no selection on the held-out event): S1 vs S1+HAND+slope vs S1+all. Paired bootstrap over chips inside events.
3. The best variant by **LOEO macro IoU** (never by test) is trained on the **official split** as an encoder x seed ensemble next to the S1-only ensemble. Threshold from valid only. Test/Bolivia IoU with bootstrap CIs and a paired test.
4. Everything is saved to `/kaggle/working/out`.

No weak-label pretraining: in the previous run it did not help (valid equal, test lower), and the weak chips would need 4,384 more EE fetches.

**Earth Engine on Kaggle (no service-account key needed).** Pick one:
1. *Interactive, simplest:* set `EE_PROJECT` as a Kaggle secret (any Cloud project of yours with the Earth Engine API enabled; a personal project outside the org also works), run the notebook interactively, and paste the code when `ee.Authenticate` asks. The fetched terrain chips are cached in `/kaggle/working/terrain`; save that folder as a Kaggle Dataset and attach it later so Save & Run All needs no login.
2. *Unattended:* run `ee.Authenticate()` once on your laptop or Colab, then store the contents of `~/.config/earthengine/credentials` as the Kaggle secret `EE_CREDENTIALS_JSON` (plus `EE_PROJECT`).
3. *Service-account key* (`EE_SERVICE_ACCOUNT_JSON`) is still supported if your org allows it.

`SMOKE=True` gives a short dry run that also tests the EE fetch.

In [ ]:
import os, sys, glob, re, json, time, random, shutil, subprocess, warnings, io, copy, math
warnings.filterwarnings("ignore")
T0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "rasterio", "earthengine-api"], check=False)

import numpy as np, pandas as pd, torch, torch.nn.functional as F
import rasterio
import segmentation_models_pytorch as smp

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", DEV, "| torch", torch.__version__, "| smp", smp.__version__)

In [ ]:
CFG = dict(
    SMOKE=False,
    EE_PROJECT=None,                         # or Kaggle secret EE_PROJECT
    # name -> terrain layers added to [VV, VH, VV-VH]
    VARIANTS={"S1": [], "S1+HAND+slope": ["hand", "slope"], "S1+all": ["jrc", "dem", "slope", "hand"]},
    LOEO=True, LOEO_ENC="unet:resnet34", LOEO_STEPS=1500, LOEO_SEEDS=[0],
    FINAL_ENCODERS=["unet:resnet34", "unet:resnext50_32x4d", "unet:mit_b2"],
    FINAL_SEEDS=[0, 1, 2], FINAL_STEPS=3000, FINAL_EXTRA_VARIANT=None,   # None = best from LOEO
    CROP=256, BS=24, LR=3e-4, WD=1e-4, EMA=0.995, EVAL_EVERY=250,
    ENC_WEIGHTS="imagenet", SIZE=512,
    EE_WORKERS=8, DEADLINE_H=10.5,
    OUT="/kaggle/working/out", ROOT="/kaggle/working/s1f11", TERR="/kaggle/working/terrain",
)
if CFG["SMOKE"]:
    CFG.update(LOEO_STEPS=30, FINAL_STEPS=30, FINAL_ENCODERS=CFG["FINAL_ENCODERS"][:1], FINAL_SEEDS=[0], EVAL_EVERY=15)
os.makedirs(CFG["OUT"], exist_ok=True); os.makedirs(CFG["TERR"], exist_ok=True)
TERRAIN_ALL = ["jrc", "dem", "slope", "hand"]
def hours(): return (time.time() - T0) / 3600
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---------- hand-labelled chips + official splits ----------
ROOT, B = CFG["ROOT"], "gs://sen1floods11/v1.1"
def sh(c):
    print("$", c); return subprocess.run(c, shell=True, capture_output=True, text=True)

if len(glob.glob(f"{ROOT}/S1Hand/*.tif")) < 446:
    if shutil.which("gsutil") is None: sh("pip install -q gsutil")
    os.makedirs(ROOT, exist_ok=True)
    sh(f"gsutil -m -q cp -r {B}/splits/flood_handlabeled {ROOT}/")
    for d in ["S1Hand", "LabelHand"]:
        print(sh(f"gsutil -m -q cp -r {B}/data/flood_events/HandLabeled/{d} {ROOT}/").stderr[-300:])

def find_csv(name):
    p = glob.glob(f"{ROOT}/**/{name}", recursive=True); assert p, f"missing {name}"; return p[0]
def read_split(name):
    df = pd.read_csv(find_csv(name), header=None)
    return [(f"{ROOT}/S1Hand/{a}", f"{ROOT}/LabelHand/{b}") for a, b in zip(df[0], df[1])]

SPLIT = {k: read_split(f"flood_{k}_data.csv") for k in ["train", "valid", "test", "bolivia"]}
SPLIT["bol"] = SPLIT.pop("bolivia")
ITEMS, IDX, pos = [], {}, 0
for k in ["train", "valid", "test", "bol"]:
    IDX[k] = np.arange(pos, pos + len(SPLIT[k])); ITEMS += SPLIT[k]; pos += len(SPLIT[k])
cid = lambda p: re.sub(r"_(S1Hand|LabelHand)\.tif$", "", os.path.basename(p))
IDS = [cid(a) for a, _ in ITEMS]
EVENT = np.array([i.split("_")[0] for i in IDS])
EVENTS = sorted(set(EVENT))
print({k: len(v) for k, v in IDX.items()}, "| events:", EVENTS)
assert len(ITEMS) == len(set(IDS)) and all(os.path.exists(a) and os.path.exists(b) for a, b in ITEMS)
if CFG["SMOKE"]: print("SMOKE: all chips are still used for terrain fetch / LOEO folds are limited below")

In [ ]:
# ---------- terrain from Earth Engine, pixel-aligned to each chip ----------
def terrain_path(i): return f"{CFG['TERR']}/{i}.npy"

def cached_terrain_dirs():
    hit = glob.glob(f"/kaggle/input/**/{IDS[0]}.npy", recursive=True)
    return os.path.dirname(hit[0]) + "/" if hit else None

missing = [i for i in IDS if not os.path.exists(terrain_path(i))]
if missing:
    alt = cached_terrain_dirs()
    if alt:
        for f in glob.glob(alt + "*.npy"): shutil.copy(f, CFG["TERR"])
        missing = [i for i in IDS if not os.path.exists(terrain_path(i))]
        print("copied terrain cache from", alt, "| still missing", len(missing))

if missing:
    import ee
    from concurrent.futures import ThreadPoolExecutor
    proj, key, cred_json = CFG["EE_PROJECT"], None, None
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        try: key = us.get_secret("EE_SERVICE_ACCOUNT_JSON")
        except Exception: key = None
        try: cred_json = us.get_secret("EE_CREDENTIALS_JSON")
        except Exception: cred_json = None
        try: proj = proj or us.get_secret("EE_PROJECT")
        except Exception: pass
    except Exception: pass
    hv = "https://earthengine-highvolume.googleapis.com"
    def ee_init(creds=None):
        last = None
        for url in (hv, None):
            try:
                if url: ee.Initialize(creds, project=proj, opt_url=url)
                else: ee.Initialize(creds, project=proj)
                return
            except Exception as e: last = e
        raise last
    try:
        if key:                                        # service-account key (blocked by some org policies)
            info = json.loads(key); proj = proj or info.get("project_id")
            ee_init(ee.ServiceAccountCredentials(info["client_email"], key_data=key))
        elif cred_json:                                # refresh token from your own account (no key creation needed)
            from google.oauth2.credentials import Credentials
            d = json.loads(cred_json)
            ee_init(Credentials(None, refresh_token=d["refresh_token"], token_uri="https://oauth2.googleapis.com/token",
                                client_id=d.get("client_id", ee.oauth.CLIENT_ID), client_secret=d.get("client_secret", ee.oauth.CLIENT_SECRET),
                                scopes=ee.oauth.SCOPES))
        else:                                          # interactive: works in an interactive session, not in Save & Run All
            try: ee_init()
            except Exception:
                ee.Authenticate(auth_mode="notebook"); ee_init()
    except Exception as e:
        raise RuntimeError("Earth Engine init failed (%s). Set EE_PROJECT and use EE_CREDENTIALS_JSON, or run this cell interactively once, or attach a terrain cache dataset." % e)

    dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
    img = ee.Image.cat([
        ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("seasonality").rename("jrc"),
        dem.rename("dem"),
        ee.Terrain.slope(dem).rename("slope"),
        ee.ImageCollection("users/gena/global-hand/hand-100").mosaic().rename("hand"),
    ]).unmask(0).toFloat()

    def chip_grid(i):
        with rasterio.open(ITEMS[IDS.index(i)][0]) as s:
            t, h, w = s.transform, s.height, s.width
        return dict(dimensions=dict(width=w, height=h), crsCode="EPSG:4326",
                    affineTransform=dict(scaleX=t.a, shearX=t.b, translateX=t.c, shearY=t.d, scaleY=t.e, translateY=t.f))

    def fetch_one(i):
        if os.path.exists(terrain_path(i)): return i, None
        err = None
        for k in range(5):
            try:
                raw = ee.data.computePixels(dict(expression=img, fileFormat="NPY", grid=chip_grid(i)))
                a = np.load(io.BytesIO(raw), allow_pickle=False)
                a = np.stack([a[n] for n in TERRAIN_ALL], 0) if a.dtype.names else np.moveaxis(a, -1, 0)
                np.save(terrain_path(i), a.astype(np.float16)); return i, None
            except Exception as e:
                err = str(e)[:200]; time.sleep(2 ** k)
        return i, err

    # test one chip first (fail fast), then the rest
    i0, e0 = fetch_one(missing[0]); assert e0 is None, f"first EE fetch failed: {e0}"
    a0 = np.load(terrain_path(i0)).astype(np.float32)
    print("first chip", i0, "shape", a0.shape, "| per-layer min/max:", [(TERRAIN_ALL[j], float(a0[j].min()), float(a0[j].max())) for j in range(4)])
    t1 = time.time(); fails = []
    with ThreadPoolExecutor(CFG["EE_WORKERS"]) as ex:
        for n, (i, e) in enumerate(ex.map(fetch_one, missing[1:]), 1):
            if e: fails.append((i, e))
            if n % 50 == 0: print(f"  fetched {n}/{len(missing) - 1} | {time.time() - t1:.0f}s | failures {len(fails)}")
    print("EE fetch done in %.0fs, failures: %d" % (time.time() - t1, len(fails)), fails[:3])
    assert len(fails) < 0.02 * len(IDS), "too many EE failures"

TERR = {}
for n, i in enumerate(IDS):
    p = terrain_path(i)
    TERR[i] = np.load(p).astype(np.float32) if os.path.exists(p) else None
nm = sum(v is None for v in TERR.values())
print("terrain chips loaded:", len(TERR) - nm, "| missing:", nm)
assert nm < 0.02 * len(IDS)

In [ ]:
# ---------- build normalised input stack: [VV, VH, VV-VH, jrc, dem, slope, hand] ----------
S = CFG["SIZE"]
def read_tif(p):
    with rasterio.open(p) as s: return s.read().astype(np.float32)

N = len(ITEMS)
XR = np.zeros((N, 7, S, S), np.float32); Y = np.zeros((N, S, S), np.int8)
for n, (a, b) in enumerate(ITEMS):
    s1 = read_tif(a)[:2]
    s1[0] = np.where(np.isfinite(s1[0]), s1[0], -10.0); s1[1] = np.where(np.isfinite(s1[1]), s1[1], -17.0)   # nodata -> typical land, not "water"
    s1 = np.clip(s1, -35.0, 5.0)
    XR[n, 0], XR[n, 1], XR[n, 2] = s1[0], s1[1], s1[0] - s1[1]
    t = TERR[IDS[n]]
    if t is None: t = np.zeros((4, S, S), np.float32)
    t = np.nan_to_num(t)
    XR[n, 3] = np.clip(t[0], 0, 12)
    XR[n, 4] = np.clip(t[1], 0, 1500)
    XR[n, 5] = np.clip(t[2], 0, 20)
    XR[n, 6] = np.log1p(np.clip(t[3], 0, 500))
    Y[n] = read_tif(b)[0].astype(np.int8)
MEAN = XR[IDX["train"]].mean((0, 2, 3), keepdims=True)[0]; STD = XR[IDX["train"]].std((0, 2, 3), keepdims=True)[0] + 1e-6
X = ((XR - MEAN[None]) / STD[None]).astype(np.float16); del XR
STD_VV, STD_VH = float(STD[0, 0, 0]), float(STD[1, 0, 0])
CHANS = {k: [0, 1, 2] + [3 + TERRAIN_ALL.index(t) for t in v] for k, v in CFG["VARIANTS"].items()}
print("X", X.shape, X.dtype, "| channels per variant:", CHANS)
print("per-channel mean", MEAN.ravel().round(2), "std", STD.ravel().round(2))
print("water share (valid px):", float((Y == 1).sum() / (Y >= 0).sum()))

In [ ]:
# ---------- model, loss, metrics ----------
def make_model(spec, in_ch):
    arch, enc = spec.split(":")
    cls = {"unet": smp.Unet, "upp": smp.UnetPlusPlus}[arch]
    w = CFG["ENC_WEIGHTS"] if CFG["ENC_WEIGHTS"] != "none" else None
    return cls(enc, encoder_weights=w, in_channels=in_ch, classes=1).to(DEV)

def masked_loss(logit, y):
    m = (y >= 0).float(); t = (y == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logit, t, reduction="none") * m).sum() / m.sum().clamp(min=1)
    p = torch.sigmoid(logit) * m
    inter = (p * t).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + (t * m).sum((1, 2, 3))
    return bce + (1 - ((2 * inter + 1) / (den + 1)).mean())

def counts(prob, y, thr=0.5):
    m = y >= 0; pr = (prob >= thr) & m; t = (y == 1) & m
    return np.stack([(pr & t).sum((1, 2)), (pr & ~t).sum((1, 2)), (~pr & t).sum((1, 2))], 1)   # (n, 3): tp, fp, fn
def iou_c(c): return float(c[:, 0].sum() / max(c.sum(), 1)) if c.ndim == 2 else float(c[0] / max(c.sum(), 1))
def prec_c(c): return float(c[:, 0].sum() / max(c[:, 0].sum() + c[:, 1].sum(), 1))
def rec_c(c): return float(c[:, 0].sum() / max(c[:, 0].sum() + c[:, 2].sum(), 1))

class EMA:
    def __init__(s, model, d): s.m = copy.deepcopy(model).eval(); s.d = d; [p.requires_grad_(False) for p in s.m.parameters()]
    @torch.no_grad()
    def update(s, model, step):
        d = min(s.d, (1 + step) / (10 + step))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

@torch.no_grad()
def predict(model, idx, chans, tta=True, bs=8):
    model.eval(); out = []
    flips = [[], [3], [2], [2, 3]] if tta else [[]]
    for i in range(0, len(idx), bs):
        x = torch.from_numpy(X[idx[i:i + bs]][:, chans].astype(np.float32)).to(DEV)
        ps = []
        for f in flips:
            xi = torch.flip(x, f) if f else x
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                o = torch.sigmoid(model(xi).float())
            ps.append(torch.flip(o, f) if f else o)
        out.append(torch.stack(ps).mean(0)[:, 0].cpu().numpy().astype(np.float16))
    return np.concatenate(out)

def make_batch(idx, chans, bs, crop, rng):
    xs, ys = [], []
    for i in rng.choice(idx, bs):
        r, c = rng.integers(0, S - crop + 1), rng.integers(0, S - crop + 1)
        x = X[i][chans, r:r + crop, c:c + crop].astype(np.float32); y = Y[i, r:r + crop, c:c + crop]
        if rng.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
        if rng.random() < .5: x, y = x[:, ::-1], y[::-1]
        k = int(rng.integers(4))
        if k: x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
        x = np.ascontiguousarray(x); sh_db = rng.uniform(-1, 1)
        x[0] += sh_db / STD_VV; x[1] += sh_db / STD_VH            # +-1 dB backscatter shift, VV-VH unchanged
        xs.append(x); ys.append(np.ascontiguousarray(y))
    return torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys).astype(np.float32))[:, None]

def train(spec, chans, train_idx, steps, seed, val_idx=None, tag=""):
    set_seed(seed); rng = np.random.default_rng(seed)
    model = make_model(spec, len(chans)); ema = EMA(model, CFG["EMA"])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["LR"], total_steps=steps, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    best, best_sd, t0, run = -1.0, None, time.time(), 0.0
    crop = min(CFG["CROP"], S)
    for step in range(1, steps + 1):
        model.train()
        x, y = make_batch(train_idx, chans, CFG["BS"], crop, rng)
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            out = model(x)
        loss = masked_loss(out.float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model, step)
        run += loss.item()
        if step % CFG["EVAL_EVERY"] == 0 or step == steps:
            msg = f"[{tag}] step {step}/{steps} loss {run / CFG['EVAL_EVERY']:.4f} ({time.time() - t0:.0f}s)"; run = 0.0
            if val_idx is not None:
                v = iou_c(counts(predict(ema.m, val_idx, chans, tta=False).astype(np.float32), Y[val_idx]))
                msg += f" valid IoU(EMA) {v:.4f}"
                if v > best: best, best_sd = v, {k: t.detach().cpu().clone() for k, t in ema.m.state_dict().items()}
            print(msg)
    if best_sd is not None: ema.m.load_state_dict(best_sd)
    del model, opt; torch.cuda.empty_cache()
    return ema.m, best

In [ ]:
# ---------- stats helpers ----------
def boot_pooled(cA, cB=None, n=2000, seed=0):
    rng = np.random.default_rng(seed); k = len(cA); a = []; d = []
    for _ in range(n):
        i = rng.integers(0, k, k); ia = iou_c(cA[i]); a.append(ia)
        if cB is not None: d.append(ia - iou_c(cB[i]))
    ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return (ci(a), (float(np.mean(d)), ci(d))) if cB is not None else ci(a)

def macro_iou(c, ev, ev_list):
    return float(np.mean([iou_c(c[ev == e]) for e in ev_list]))

def boot_macro(cA, cB, ev, ev_list, n=2000, seed=0):
    rng = np.random.default_rng(seed); groups = {e: np.where(ev == e)[0] for e in ev_list}; da = []; dd = []
    for _ in range(n):
        ma, mb = [], []
        for e in ev_list:
            g = rng.choice(groups[e], len(groups[e])); ma.append(iou_c(cA[g])); mb.append(iou_c(cB[g]))
        da.append(np.mean(ma)); dd.append(np.mean(ma) - np.mean(mb))
    ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return float(np.mean(dd)), ci(dd)

In [ ]:
# ---------- stage 1: leave-one-event-out screening ----------
LOEO_RES = {}      # variant -> dict(counts=(N,3) at thr 0.5 over all 446 chips, seeds avg handled by pooling counts per seed)
BEST = None
if CFG["LOEO"]:
    events = EVENTS[:2] if CFG["SMOKE"] else EVENTS
    allidx = np.arange(N)
    for v, chans in CHANS.items():
        cnt = {s: np.zeros((N, 3), np.int64) for s in CFG["LOEO_SEEDS"]}; held_mask = np.zeros(N, bool)
        for e in events:
            if hours() > CFG["DEADLINE_H"] - 5.0: print("time guard: stopping LOEO early"); break
            held = np.where(EVENT == e)[0]; tr = np.where(EVENT != e)[0]; held_mask[held] = True
            for s in CFG["LOEO_SEEDS"]:
                m, _ = train(CFG["LOEO_ENC"], chans, tr, CFG["LOEO_STEPS"], s, None, tag=f"LOEO {v}|{e}|s{s}")
                p = predict(m, held, chans, tta=False).astype(np.float32); cnt[s][held] = counts(p, Y[held], 0.5)
                print(f"  ==> {v} | held-out {e}: IoU {iou_c(cnt[s][held]):.3f} precision {prec_c(cnt[s][held]):.3f} recall {rec_c(cnt[s][held]):.3f} | elapsed {hours():.2f} h")
                del m
        LOEO_RES[v] = dict(counts=np.mean([cnt[s] for s in CFG["LOEO_SEEDS"]], 0), mask=held_mask, events=[e for e in events if (EVENT == e).any()])
    ev_done = [e for e in events if all(LOEO_RES[v]["mask"][EVENT == e].all() for v in LOEO_RES)]
    msk = np.isin(EVENT, ev_done)
    rows = []
    for v in LOEO_RES:
        c = LOEO_RES[v]["counts"]
        per = {e: iou_c(c[EVENT == e]) for e in ev_done}
        pe = {e: prec_c(c[EVENT == e]) for e in ev_done}
        rows.append(dict(variant=v, macro_IoU=np.mean(list(per.values())), macro_precision=np.mean(list(pe.values())), worst_event_IoU=min(per.values()), worst_event=min(per, key=per.get), **{f"IoU_{e}": per[e] for e in ev_done}))
    LOEO_DF = pd.DataFrame(rows).set_index("variant")
    pd.set_option("display.width", 250, "display.max_columns", 40)
    print("\nLOEO (chip counts at threshold 0.5, seed-averaged), events:", ev_done)
    print(LOEO_DF.round(3).T.to_string())
    base = "S1" if "S1" in LOEO_RES else list(LOEO_RES)[0]
    for v in LOEO_RES:
        if v == base: continue
        d, ci = boot_macro(LOEO_RES[v]["counts"][msk], LOEO_RES[base]["counts"][msk], EVENT[msk], ev_done)
        print(f"paired macro-IoU diff {v} minus {base}: {d:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}] (chips resampled within events; ignores seed noise)")
    BEST = str(LOEO_DF["macro_IoU"].idxmax()); LOEO_DF.to_csv(f"{CFG['OUT']}/loeo_by_event.csv")
    print("\nbest variant by LOEO macro IoU:", BEST)
else:
    BEST = CFG["FINAL_EXTRA_VARIANT"] or list(CHANS)[-1]
if CFG["FINAL_EXTRA_VARIANT"]: BEST = CFG["FINAL_EXTRA_VARIANT"]

In [ ]:
# ---------- stage 2: official split, encoder x seed ensembles ----------
FINAL_VARIANTS = ["S1"] if BEST == "S1" else ["S1", BEST]
FINAL = []   # dicts: variant, enc, seed, probs{valid,test,bol}
for v in FINAL_VARIANTS:
    for enc in CFG["FINAL_ENCODERS"]:
        for s in CFG["FINAL_SEEDS"]:
            if hours() > CFG["DEADLINE_H"]: print("time guard: skipping", v, enc, s); continue
            tag = f"{v}|{enc}|s{s}"
            try: m, bv = train(enc, CHANS[v], IDX["train"], CFG["FINAL_STEPS"], s, IDX["valid"], tag=tag)
            except Exception as e: print("skip", tag, type(e).__name__, str(e)[:160]); continue
            probs = {k: predict(m, IDX[k], CHANS[v]) for k in ["valid", "test", "bol"]}
            r = dict(variant=v, enc=enc, seed=s, best_valid=bv, probs=probs)
            for k in ["test", "bol"]: r[k + "_iou"] = iou_c(counts(probs[k].astype(np.float32), Y[IDX[k]]))
            print(f"==> {tag}: valid {bv:.4f} | test {r['test_iou']:.4f} | bolivia {r['bol_iou']:.4f} | elapsed {hours():.2f} h")
            torch.save({k: t.half() if t.dtype.is_floating_point else t for k, t in m.state_dict().items()}, f"{CFG['OUT']}/{v.replace('+', '_')}_{enc.replace(':', '_')}_s{s}.pt")
            FINAL.append(r); del m; torch.cuda.empty_cache()
assert FINAL, "no final model trained"

In [ ]:
# ---------- results ----------
def ens(runs, split): return np.mean([r["probs"][split].astype(np.float32) for r in runs], 0)
def best_thr(runs):
    pv = ens(runs, "valid"); grid = np.arange(0.30, 0.71, 0.02)
    return float(grid[int(np.argmax([iou_c(counts(pv, Y[IDX["valid"]], t)) for t in grid]))])

def report(name, runs, thr=None):
    thr = best_thr(runs) if thr is None else thr; row = dict(model=name, n=len(runs), thr=round(thr, 2))
    for sp in ["valid", "test", "bol"]:
        c = counts(ens(runs, sp), Y[IDX[sp]], thr); row[sp] = iou_c(c)
        if sp != "valid": row[sp + "_ci95"] = tuple(round(x, 3) for x in boot_pooled(c)); row[sp + "_prec"] = prec_c(c); row[sp + "_rec"] = rec_c(c)
    return row

rows = []
for v in FINAL_VARIANTS:
    for enc in CFG["FINAL_ENCODERS"]:
        sub = [r for r in FINAL if r["variant"] == v and r["enc"] == enc]
        if sub: rows.append(report(f"{v} | {enc} x{len(sub)}", sub))
    sub = [r for r in FINAL if r["variant"] == v]
    rows.append(report(f"{v} | ALL ensemble", sub))
RES = pd.DataFrame(rows)
pd.set_option("display.width", 260, "display.max_colwidth", 50, "display.max_columns", 30)
print(RES.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

singles = pd.DataFrame([{k: r[k] for k in ["variant", "enc", "seed", "best_valid", "test_iou", "bol_iou"]} for r in FINAL])
print("\nsingle models:\n", singles.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

if BEST != "S1":
    a = [r for r in FINAL if r["variant"] == BEST]; b = [r for r in FINAL if r["variant"] == "S1"]
    ta, tb = best_thr(a), best_thr(b)
    ca, cb = counts(ens(a, "test"), Y[IDX["test"]], ta), counts(ens(b, "test"), Y[IDX["test"]], tb)
    _, (dm, dci) = boot_pooled(ca, cb)
    print(f"\nTEST paired (same chips): {BEST} ensemble {iou_c(ca):.4f} vs S1 ensemble {iou_c(cb):.4f} | diff {dm:+.4f} 95% CI [{dci[0]:+.4f}, {dci[1]:+.4f}]")
    wins = sum(x["test_iou"] > np.mean([y["test_iou"] for y in b if y["enc"] == x["enc"]]) for x in a)
    print(f"single {BEST} models above the matching S1 encoder mean: {wins}/{len(a)}")
print("\nReference: DeepSARFlood reports 0.72 test IoU with 6 inputs (VV, VH, JRC seasonality, DEM, slope, HAND); label setup and metric there may differ from this notebook.")

RES.to_csv(f"{CFG['OUT']}/official_split_ensembles.csv", index=False); singles.to_csv(f"{CFG['OUT']}/single_models.csv", index=False)
json.dump(dict(cfg={k: v for k, v in CFG.items()}, best_variant=BEST, elapsed_h=hours()), open(f"{CFG['OUT']}/run.json", "w"), indent=1, default=str)
print("saved to", CFG["OUT"], "| total %.2f h" % hours())

## How to read it
- **LOEO table first.** It is the cross-event number (the one that matters operationally). A variant only "helps" if its paired macro-IoU interval stays above 0 and the per-event table shows the gain on the hard events (Somalia, Pakistan, Spain), not just on easy ones.
- **Official-split table** places the ensembles on the same footing as published results. The test set has only 90 chips, so intervals are wide (about +-0.1); trust the paired difference, not the point values.
- If S1+terrain wins on LOEO but not on the official split, say so: terrain may help generalisation to new events without changing the within-event benchmark.
- Elevation is an absolute value and can act as a location shortcut; that is why `S1+HAND+slope` (relative terrain only) is screened separately from `S1+all`.